# Disease Prediction System (Healthcare Analytics)

**Objective:** Predict diseases (e.g., diabetes or heart disease) using patient medical data to assist doctors in early diagnosis.

This notebook covers the complete machine learning pipeline:
1. Data Collection (Synthetic Generation)
2. Data Preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Model Selection & Training
6. Model Evaluation
7. Disease Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')
%matplotlib inline

## Step 1: Data Collection
Since healthcare datasets with specific symptoms and vitals are often hard to find in a single CSV, we will generate a robust synthetic dataset that matches the project specifications.

In [ ]:
np.random.seed(42)
random.seed(42)

def generate_healthcare_data(num_samples=2500):
    data = []
    diseases = ['Flu', 'Heart Disease', 'Diabetes', 'Hypertension', 'Common Cold', 'Asthma', 'Healthy']
    
    for i in range(1, num_samples + 1):
        patient_id = f"P{i:04d}"
        disease = random.choice(diseases)
        
        if disease == 'Flu':
            age = random.randint(5, 80)
            gender = random.choice(['Male', 'Female'])
            symptoms = random.sample(['Fever', 'Cough', 'Fatigue', 'Headache', 'Sore Throat', 'Muscle Ache'], random.randint(2, 4))
            bp = random.randint(100, 130)
            sugar = random.randint(80, 110)
            cholesterol = random.randint(150, 220)
            history = random.choice(['None', 'Asthma'])
            
        elif disease == 'Heart Disease':
            age = random.randint(45, 90)
            gender = random.choice(['Male', 'Female', 'Male'])
            symptoms = random.sample(['Chest Pain', 'Shortness of Breath', 'Fatigue', 'Dizziness', 'Palpitations'], random.randint(2, 4))
            bp = random.randint(130, 180)
            sugar = random.randint(90, 150)
            cholesterol = random.randint(200, 300)
            history = random.choice(['Hypertension', 'Diabetes', 'None', 'Hypertension'])
            
        elif disease == 'Diabetes':
            age = random.randint(30, 85)
            gender = random.choice(['Male', 'Female'])
            symptoms = random.sample(['Increased Thirst', 'Frequent Urination', 'Fatigue', 'Blurred Vision', 'Weight Loss'], random.randint(2, 4))
            bp = random.randint(110, 150)
            sugar = random.randint(150, 350)
            cholesterol = random.randint(180, 260)
            history = random.choice(['None', 'Hypertension', 'Obesity'])
            
        elif disease == 'Hypertension':
            age = random.randint(35, 90)
            gender = random.choice(['Male', 'Female'])
            symptoms = random.sample(['Headache', 'Shortness of Breath', 'Nosebleeds', 'Dizziness', 'None'], random.randint(1, 3))
            bp = random.randint(140, 200)
            sugar = random.randint(80, 130)
            cholesterol = random.randint(180, 250)
            history = random.choice(['None', 'Diabetes', 'Obesity'])
            
        elif disease == 'Common Cold':
            age = random.randint(2, 80)
            gender = random.choice(['Male', 'Female'])
            symptoms = random.sample(['Runny Nose', 'Sore Throat', 'Cough', 'Congestion', 'Sneezing'], random.randint(2, 4))
            bp = random.randint(90, 130)
            sugar = random.randint(70, 110)
            cholesterol = random.randint(140, 220)
            history = random.choice(['None', 'Asthma'])
            
        elif disease == 'Asthma':
            age = random.randint(5, 70)
            gender = random.choice(['Male', 'Female'])
            symptoms = random.sample(['Shortness of Breath', 'Wheezing', 'Chest Tightness', 'Cough'], random.randint(2, 3))
            bp = random.randint(100, 130)
            sugar = random.randint(80, 120)
            cholesterol = random.randint(150, 230)
            history = random.choice(['None', 'Allergies'])
            
        else: # Healthy
            age = random.randint(18, 65)
            gender = random.choice(['Male', 'Female'])
            symptoms = ['None'] if random.random() > 0.1 else [random.choice(['Headache', 'Fatigue'])]
            bp = random.randint(100, 120)
            sugar = random.randint(70, 100)
            cholesterol = random.randint(120, 200)
            history = 'None'

        symptoms_str = ", ".join(symptoms)
        
        # Introduce missing values (~3%)
        if random.random() < 0.03: bp = np.nan
        if random.random() < 0.03: sugar = np.nan
        if random.random() < 0.03: cholesterol = np.nan
            
        data.append({
            'Patient_ID': patient_id,
            'Age': age,
            'Gender': gender,
            'Symptoms': symptoms_str,
            'Blood_Pressure': bp,
            'Sugar_Level': sugar,
            'Cholesterol': cholesterol,
            'Medical_History': history,
            'Disease': disease
        })
        
    return pd.DataFrame(data)

# Generate and save
df_raw = generate_healthcare_data(3000)
df_raw.to_csv('healthcare_dataset.csv', index=False)
print("Dataset generated and saved to 'healthcare_dataset.csv'")
df = pd.read_csv('healthcare_dataset.csv')
df.head()

## Step 2 & 4: Data Preprocessing and Feature Engineering
Handling missing values, encoding categorical variables, and converting symptoms into numerical form (one-hot encoding).

In [ ]:
# 1. Handle Missing Values
print("Missing values before:")
print(df.isnull().sum())

df['Blood_Pressure'] = df['Blood_Pressure'].fillna(df['Blood_Pressure'].median())
df['Sugar_Level'] = df['Sugar_Level'].fillna(df['Sugar_Level'].median())
df['Cholesterol'] = df['Cholesterol'].fillna(df['Cholesterol'].median())

# 2. Encode categorical variables
label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])

# One-Hot Encoding for Medical History
df = pd.get_dummies(df, columns=['Medical_History'], drop_first=True)

# 3. Feature Engineering: Convert Symptoms into numerical form (One-Hot Encoding)
# Split the comma-separated symptoms and create dummy variables
symptoms_dummies = df['Symptoms'].str.get_dummies(sep=', ')
df_processed = pd.concat([df, symptoms_dummies], axis=1)
df_processed.drop(['Symptoms', 'Patient_ID'], axis=1, inplace=True)

print("\nShape of processed dataset:", df_processed.shape)
df_processed.head()

## Step 3: Exploratory Data Analysis (EDA)
Visualizing relationships in the data.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Disease', order=df['Disease'].value_counts().index, palette='viridis')
plt.title('Distribution of Diseases')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Disease', y='Age', palette='Set2')
plt.title('Age Distribution across Diseases')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Disease', y='Blood_Pressure', palette='Set3')
plt.title('Blood Pressure across Diseases')
plt.xticks(rotation=45)
plt.show()

## Step 5: Model Selection and Training
We will test multiple algorithms: Logistic Regression, Decision Tree, Random Forest, Naive Bayes, and SVM.

In [ ]:
X = df_processed.drop('Disease', axis=1)
y = df_processed['Disease']

# Scale numerical features
scaler = StandardScaler()
numerical_cols = ['Age', 'Blood_Pressure', 'Sugar_Level', 'Cholesterol']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Support Vector Machine': SVC(kernel='linear', random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} trained. Accuracy: {accuracy:.4f}")

best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
print(f"\nBest Model: {best_model_name} with Accuracy {results[best_model_name]:.4f}")

## Step 8: Model Evaluation
Detailed metrics for the best model.

In [ ]:
y_pred_best = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred_best))

# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=best_model.classes_, yticklabels=best_model.classes_)
plt.xlabel('Predicted Disease')
plt.ylabel('Actual Disease')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()

## Step 7: Disease Prediction (Inference)
Function to predict disease based on new patient symptoms and vitals.

In [ ]:
def predict_disease(patient_data, model, scaler, X_columns, label_encoder):
    """
    patient_data: dict containing Age, Gender, Symptoms (list), Blood_Pressure, Sugar_Level, Cholesterol, Medical_History
    """
    # Create a DataFrame with a single row initialized to 0 for all model features
    df_input = pd.DataFrame(0, index=[0], columns=X_columns)
    
    # 1. Fill Numerical Features
    df_input['Age'] = patient_data['Age']
    df_input['Blood_Pressure'] = patient_data['Blood_Pressure']
    df_input['Sugar_Level'] = patient_data['Sugar_Level']
    df_input['Cholesterol'] = patient_data['Cholesterol']
    
    # 2. Scale Numerical Features
    numerical_cols = ['Age', 'Blood_Pressure', 'Sugar_Level', 'Cholesterol']
    df_input[numerical_cols] = scaler.transform(df_input[numerical_cols])
    
    # 3. Categorical Encoding (Gender)
    try:
        encoded_gender = label_encoder.transform([patient_data['Gender']])[0]
    except:
        encoded_gender = 0 # Default if unknown
    if 'Gender' in df_input.columns: df_input['Gender'] = encoded_gender
        
    # 4. Medical History Dummies
    history_col = f"Medical_History_{patient_data['Medical_History']}"
    if history_col in df_input.columns:
        df_input[history_col] = 1
        
    # 5. Symptoms Dummies
    for symptom in patient_data['Symptoms']:
        if symptom in df_input.columns:
            df_input[symptom] = 1
            
    # Predict
    prediction = model.predict(df_input)[0]
    
    return prediction

# --- TEST THE PREDICTION SYSTEM ---
new_patient = {
    'Age': 55,
    'Gender': 'Male',
    'Symptoms': ['Chest Pain', 'Shortness of Breath'],
    'Blood_Pressure': 160,
    'Sugar_Level': 110,
    'Cholesterol': 260,
    'Medical_History': 'Hypertension'
}

predicted_disease = predict_disease(new_patient, best_model, scaler, X.columns, label_encoder)
print("Patient Data:", new_patient)
print("\n>> Predicted Disease:", predicted_disease)
